# Personal Writeup

**U.S. Electricity Dashboard** <br>
Purple Flamingo - Aileen Yang <br>


**WHAT WE BUILT** <br>
For our Advanced Computing for Policy final project, my team built a four-page interactive dashboard tracking U.S. electricity demand and wholesale prices in near-real-time. The app pulls daily fuel-type and regional demand data from the EIA API, combines it with hourly locational marginal prices (LMP) from CAISO, NYISO, and ERCOT via the Gridstatus library, stages everything through Google BigQuery, and serves it through a Streamlit frontend.

The Streamlit pages cover: electricity demand by fuel type, demand by region with anomaly detection, a grid stress analysis page that flags statistically unusual demand days, and a price-demand correlation view that lets you see how wholesale electricity prices move relative to consumption across three major grid operators.

![architect](diagram.png)

**THE PROPOSAL** <br>
Our project started from a single research question: how does electricity demand and the generation mix change during periods of unusually high or low demand? 

The original plan was ambitious: analyze demand across U.S. regions, study fuel mix shifts, compare interregional grid behavior, integrate weather data from NOAA, and use high-frequency EIA data throughout. 

We quickly realized the scope was too broad for the timeline. Managing multiple complex datasets introduced performance concerns, and we didn't yet have a rigorous definition of what "grid stress" actually meant in measurable terms. Those early tensions forced us to be deliberate about what we were actually trying to answer, which turned out to be a more valuable exercise than if the proposal had gone smoothly.

The project went through three meaningful evolutions before we arrived at the final system. 
First, we narrowed the scope. We dropped weather integration entirely, removed the exploratory framing, and locked in on one concrete analytical goal: characterizing demand behavior during extreme periods. 
Second, we upgraded the architecture. What started as direct API calls from the app became a proper data pipeline: two ETL loader scripts ingest data on a schedule, stage it into Google BigQuery, and the Streamlit frontend reads exclusively from there. That shift from local API calls to a cloud warehouse solved our performance concerns and made the system genuinely scalable. 
Third, we resolved the methodology gap by formalizing "grid stress" as a statistical concept: days where total demand deviates beyond a configurable z-score threshold are flagged as anomalies, which gave us a reproducible, tunable definition that works across regions and time windows.

**THE CODE** <br>
The final system is a Streamlit dashboard covering fuel-type demand, regional demand with anomaly detection, a grid stress analysis page, and a price–demand correlation view joining EIA data with hourly locational marginal prices from CAISO, NYISO, and ERCOT via Gridstatus. A few engineering decisions stand out. 

The EIA loader handles both fuel and regional pipelines from a single script, controlled by a DATA_SOURCE environment variable so there is no duplicated code, just one flag flipped at runtime. 

The Gridstatus pipeline wraps each ISO extractor in a try/except loop so a single API failure never blocks the others from loading. The price–demand join required building an explicit crosswalk dictionary between the two sources, which use different identifiers for the same grid operators (CAISO vs CISO). 

Across the entire codebase, we used Pandera to define explicit schemas for every major dataframe shape — but critically, validation failures are caught and logged as warnings rather than exceptions, so the app never crashes on a malformed upstream row. 

There is also a credential layer tries a service account key first and falls back to local ADC, meaning the same code runs on Streamlit Cloud and on our laptops without any configuration change. 

![anomaly](Anomaly.png)

![pricedemand](Price-demand.png)

**WHAT I LEARNED** <br>
Going into this project, I assumed my CS background would make the technical side straightforward and that the interesting challenge would be on the analytical or policy side, but it turned out to be the opposite. 

The hardest problems weren't algorithmic; they were about trust and representation. A question that I constantly ask myself a question: how can we build a system where you can actually believe the numbers it shows us? And more subtly: once we can believe them, how do we decide which numbers to show at all? 

That second question is harder than it looks. A dashboard for policymakers shouldn't be a data display. We spent real time debating whether z-score was the right framing for "grid stress," because the threshold you pick determines which days get flagged, and flagged days are the ones that drive decisions. 

It's simultaneously a statistical choice and a policy choice. That tension, between the precision that technical systems project and the interpretive judgments quietly embedded inside them, is something I hadn't fully appreciated before building something where the output was meant to inform real decisions. It pushed me toward a more honest understanding of what data tools actually do: they don't surface objective truth, they structure attention. 

Getting that structure right urned out to be the most intellectually demanding part of the project, and the part I'll think about longest.